In [ ]:
# EM08DS Final Project - Analysis Pipeline
# Predicting Trust in AI-Based Decision-Making from survey features using
# Logistic Regression and Random Forest classifiers.

"""
Given the small sample size (N=45), Leave-One-Out Cross-Validation (LOOCV)
is used instead of a single train/test split to obtain a more stable
estimate of generalisation performance.
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

COLOR_PRIMARY = "#2C5F8A"
COLOR_SECOND = "#C9772C"
COLOR_GRID = "#E5E5E5"

df = pd.read_csv("data/ai_trust_survey_synthetic.csv")

# --- Feature engineering -------------------------------------------------
le_tech = LabelEncoder()
le_age = LabelEncoder()
le_usage = LabelEncoder()
le_edu = LabelEncoder()

age_order = ["18-24", "25-34", "35-44", "45+"]
usage_order = ["Never", "Rarely", "Weekly", "Daily"]

df["age_ordinal"] = df["age_group"].apply(lambda x: age_order.index(x))
df["usage_ordinal"] = df["usage_frequency"].apply(lambda x: usage_order.index(x))
df["technical_binary"] = (df["technical_background"] == "Yes").astype(int)

feature_cols = [
    "ai_familiarity_1to5",
    "age_ordinal",
    "usage_ordinal",
    "technical_binary",
    "q3_perceived_transparency",
    "q5_bias_concern",
]

X = df[feature_cols].values
y = (df["trust_label"] == "High Trust").astype(int).values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

loo = LeaveOneOut()

# --- Logistic Regression --------------------------------------------------
logreg = LogisticRegression(max_iter=1000, random_state=42)
y_pred_lr = cross_val_predict(logreg, X_scaled, y, cv=loo)

# --- Random Forest ---------------------------------------------------------
rf = RandomForestClassifier(n_estimators=300, max_depth=4, random_state=42)
y_pred_rf = cross_val_predict(rf, X, y, cv=loo)

results = {}
for name, y_pred in [("Logistic Regression", y_pred_lr), ("Random Forest", y_pred_rf)]:
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    results[name] = {"accuracy": acc, "f1": f1}
    print(f"\n=== {name} (LOOCV) ===")
    print(f"Accuracy: {acc:.3f}")
    print(f"F1-score: {f1:.3f}")
    print(classification_report(y, y_pred, target_names=["Low Trust", "High Trust"]))

# Fit RF on full data for feature importance (descriptive, not evaluative)
rf_full = RandomForestClassifier(n_estimators=300, max_depth=4, random_state=42)
rf_full.fit(X, y)
importances = pd.Series(rf_full.feature_importances_, index=feature_cols).sort_values()

feature_labels = {
    "ai_familiarity_1to5": "AI Familiarity",
    "age_ordinal": "Age Group",
    "usage_ordinal": "AI Usage Frequency",
    "technical_binary": "Technical Background",
    "q3_perceived_transparency": "Perceived Transparency",
    "q5_bias_concern": "Bias Concern",
}
importances.index = [feature_labels[i] for i in importances.index]

# --- Figure 1: Model comparison (Accuracy & F1) ---------------------------
fig, ax = plt.subplots(figsize=(6.5, 4))
models = list(results.keys())
acc_vals = [results[m]["accuracy"] for m in models]
f1_vals = [results[m]["f1"] for m in models]
x = np.arange(len(models))
width = 0.32
ax.bar(x - width/2, acc_vals, width, label="Accuracy", color=COLOR_PRIMARY)
ax.bar(x + width/2, f1_vals, width, label="F1-score", color=COLOR_SECOND)
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 1.0)
ax.set_ylabel("Score")
ax.set_title("Figure 1. Model Performance Comparison (LOOCV, N=45)")
ax.grid(axis="y", color=COLOR_GRID)
ax.set_axisbelow(True)
ax.legend(frameon=False)
for i, v in enumerate(acc_vals):
    ax.text(i - width/2, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)
for i, v in enumerate(f1_vals):
    ax.text(i + width/2, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig("figures/fig1_model_comparison.png", dpi=200)
plt.close()

# --- Figure 2: Feature importance (Random Forest) --------------------------
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.barh(importances.index, importances.values, color=COLOR_PRIMARY)
ax.set_xlabel("Relative Importance (Gini)")
ax.set_title("Figure 2. Random Forest Feature Importance")
ax.grid(axis="x", color=COLOR_GRID)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("figures/fig2_feature_importance.png", dpi=200)
plt.close()

# --- Figure 3: Confusion matrix (Random Forest, best-performing model) ----
cm = confusion_matrix(y, y_pred_rf)
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Low Trust", "High Trust"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Low Trust", "High Trust"])
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Figure 3. Confusion Matrix - Random Forest (LOOCV)")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=13)
plt.tight_layout()
plt.savefig("figures/fig3_confusion_matrix.png", dpi=200)
plt.close()

# --- Save summary stats for the report -------------------------------------
summary = {
    "n": len(df),
    "logreg_accuracy": results["Logistic Regression"]["accuracy"],
    "logreg_f1": results["Logistic Regression"]["f1"],
    "rf_accuracy": results["Random Forest"]["accuracy"],
    "rf_f1": results["Random Forest"]["f1"],
    "top_feature": importances.index[-1],
    "top_feature_importance": importances.values[-1],
}

pd.Series(summary).to_csv("data/model_summary.csv")
print("\nSummary:", summary)

FileNotFoundError: [Errno 2] No such file or directory: 'data/ai_trust_survey_synthetic.csv'